# Final Demo: Dataset Streaming Simulation

This notebook is the final presentation layer for the dataset-only demo. It uses downloaded ARCO-ERA5 weather and BTS flight outcomes, then replays real Gold feature rows as a streaming test-set simulation through Kafka and the deployed prediction API.

All prediction evidence comes from the downloaded project dataset and generated lakehouse outputs.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import textwrap

PROJECT_ROOT = Path('/workspace') if Path('/workspace').exists() else Path.cwd().resolve()
STREAM_DIR = PROJECT_ROOT / 'data/local_cache/streaming_predictions'
STREAM_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Streaming evidence dir:', STREAM_DIR)

## 1. Helper Functions

These helpers keep command output visible in the notebook while preserving commands exactly as they can be rerun from the VM or Jupyter container.

In [ ]:
def run(command: str, timeout: int = 120) -> str:
    print('$', command)
    completed = subprocess.run(
        command,
        shell=True,
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')
    return completed.stdout


def tail_jsonl(path: Path, limit: int = 5):
    if not path.exists():
        print('Missing JSONL:', path)
        return []
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    return rows[-limit:]

## 2. Service Health

The final demo needs MinIO, Kafka, Spark, MLflow, BentoML API, Prometheus, Grafana, and optionally the LLM chat UI.

In [ ]:
run('docker compose ps', timeout=60)

In [ ]:
import requests

checks = {
    'aviation-api': 'http://aviation-api:3000/metrics',
    'prometheus': 'http://prometheus:9090/-/ready',
    'grafana': 'http://grafana:3000/api/health',
    'kafka-ui': 'http://kafka-ui:8080',
    'mlflow': 'http://mlflow:5000',
}

for name, url in checks.items():
    try:
        response = requests.get(url, timeout=10)
        print(f'{name}: HTTP {response.status_code}')
    except Exception as error:
        print(f'{name}: {error}')

## 3. API Prediction With A Real Gold Row

This proves that the deployed BentoML API can score a real feature row produced by the Spark Gold table.

In [ ]:
run('python -m spark_jobs.call_api_with_gold_sample --year 2024 --month 1 --api-url http://aviation-api:3000/predict', timeout=180)

## 4. Dataset Streaming Replay

This is the main streaming proof. The script reads real Gold feature rows from the downloaded lakehouse, publishes each event to Kafka topic `simulation.prediction.requests`, calls the deployed API, and writes JSONL evidence for review.

In [ ]:
simulation_path = STREAM_DIR / 'notebook_gold_simulation.jsonl'
if simulation_path.exists():
    simulation_path.unlink()

cmd = (
    'python -m api.simulate_gold_stream_predict '
    '--year 2024 '
    '--month 1 '
    '--limit 25 '
    '--delay-seconds 0.1 '
    f'--output-jsonl {simulation_path} '
    '--api-url http://aviation-api:3000/predict'
)
run(cmd, timeout=300)

In [ ]:
events = tail_jsonl(simulation_path, limit=5)
for event in events:
    source = event.get('source_event', {})
    response = event.get('api_response', {})
    print({
        'sequence': event.get('sequence'),
        'flight_date': source.get('flight_date'),
        'route': f"{source.get('origin')}->{source.get('destination')}",
        'actual_label': source.get('label'),
        'prediction': response.get('prediction'),
        'risk_band': response.get('risk_band'),
        'probability': response.get('disruption_probability'),
    })

## 5. Kafka Topic Evidence

Open Kafka UI at `http://localhost:8085` and show topic `simulation.prediction.requests`. The command below is a CLI check for the same topic.

In [ ]:
run('docker compose exec -T kafka /opt/kafka/bin/kafka-topics.sh --bootstrap-server kafka:9092 --list | sort', timeout=60)

## 6. 10x API Load Test

This demonstrates the required concurrent API stability check.

In [ ]:
run('python3 api/load_test.py --url http://aviation-api:3000/predict --requests 100 --concurrency 10', timeout=180)

## 7. Prometheus Metrics

These queries confirm that Prometheus observes prediction requests, source labels, latency, and failures.

In [ ]:
def promql(query: str):
    response = requests.get('http://prometheus:9090/api/v1/query', params={'query': query}, timeout=10)
    response.raise_for_status()
    return response.json()['data']['result']

queries = {
    'requests_by_source_and_risk': 'sum by (source, risk_band) (aviation_prediction_requests_by_source_total)',
    'failed_requests': 'sum(bentoml_service_request_total{http_response_code!="200"})',
    'p95_latency_seconds': 'histogram_quantile(0.95, sum(rate(aviation_prediction_latency_seconds_bucket[1m])) by (le))',
}

for name, query in queries.items():
    print('---', name, '---')
    print(json.dumps(promql(query), indent=2))

## 8. Grafana Dashboards

Open Grafana at `http://localhost:3001` and show:

- `Aviation Prediction API` for API request rate, latency, failures, and risk bands.
- `Aviation Dataset Simulation Stream` for replay event volume and simulated risk-band split.

## 9. LLM Project Q&A Assistant

The optional LLM layer uses Groq Llama 3.3 70B when `GROQ_API_KEY` is configured. It answers questions from dataset replay evidence and Prometheus metrics. If the key is missing, it returns a local fallback summary.

In [ ]:
question = 'Explain what is happening in this aviation dataset streaming demo right now. Mention the downloaded datasets, replay simulation, risk predictions, and limitations.'
cmd = (
    'python -m api.llm_ops_assistant '
    '--prometheus-url http://prometheus:9090 '
    f'--simulation-jsonl {simulation_path} '
    f'--question {json.dumps(question)}'
)
run(cmd, timeout=180)

## 10. Browser Chat UI

Start the browser assistant with `docker compose up -d llm-chat`, then open `http://localhost:7860`. Ask:

- `How does the dataset streaming replay work?`
- `Which downloaded datasets are used by the model?`
- `What should I show in Grafana for final presentation?`

In [ ]:
try:
    response = requests.get('http://llm-chat:7860/api/health', timeout=10)
    print(response.json())
    print('Browser URL: http://localhost:7860')
except Exception as error:
    print('LLM chat UI is not running yet. Start it with: docker compose up -d llm-chat')
    print(error)

## Final Talking Points

- The model is trained from downloaded ARCO-ERA5 weather and BTS flight outcomes.
- The final demo replays real Gold feature rows as a streaming test-set simulation.
- Kafka shows event movement, BentoML scores each event, and Grafana/Prometheus show operational evidence.
- The LLM assistant is a Q&A layer over project evidence; it is not used for model training or scoring.